# Level 2 — Task 3: Fraud Detection

## Objective
Build a machine-learning system to identify potentially fraudulent credit-card transactions.

## Dataset
This project uses the public **Credit Card Fraud Detection** dataset by ULB/Worldline, commonly distributed through Kaggle.

Dataset file expected:
`data/creditcard.csv`

The dataset contains anonymized transaction features, `Time`, `Amount`, and the binary target `Class`:
- `0` = legitimate transaction
- `1` = fraud

> **Important:** Fraud detection is highly imbalanced. Accuracy alone can be misleading, so this project focuses on precision, recall, F1-score and Precision-Recall AUC.

## Workflow
1. Load and inspect the transaction dataset.
2. Analyse the severe class imbalance.
3. Check missing values and duplicates.
4. Scale `Amount` and use the anonymized PCA-style features.
5. Split the data with stratification.
6. Train Logistic Regression with class weighting.
7. Train Random Forest as a nonlinear comparison.
8. Evaluate with confusion matrices, precision, recall, F1 and PR-AUC.
9. Inspect Random Forest feature importance.
10. Explain practical fraud-detection trade-offs.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_auc_score
)

DATA_PATH = "../data/creditcard.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "creditcard.csv was not found in ../data/. "
        "Download the ULB Credit Card Fraud Detection dataset and place creditcard.csv there."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 1. Initial inspection


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass counts:")
display(df["Class"].value_counts())


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
fraud_rate = df["Class"].mean() * 100

plt.figure(figsize=(7, 5))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values)
plt.title("Legitimate vs Fraudulent Transactions")
plt.xlabel("Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Transactions")
plt.show()

print(f"Fraud rate: {fraud_rate:.4f}%")


**Observation:** Fraud is a very small minority class. A model that predicts every transaction as legitimate could still achieve very high accuracy while detecting zero fraud. Therefore, fraud recall and precision are more informative than accuracy alone.


## 2. Transaction amount analysis


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="Amount", hue="Class", bins=60, element="step", stat="density", common_norm=False)
plt.xlim(0, df["Amount"].quantile(0.99))
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Density")
plt.show()

display(df.groupby("Class")["Amount"].describe().T)


## 3. Prepare features

`Class` is the target. `Amount` is standardized because it is on a different numerical scale from the anonymized `V1`–`V28` variables. `Time` is retained as a predictor.

The train/test split is stratified so the rare fraud class is represented in both sets.


In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Fraud in train:", int(y_train.sum()))
print("Fraud in test:", int(y_test.sum()))


## 4. Logistic Regression

Class weighting is used to penalize errors on the minority fraud class more strongly.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

log_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, log_pred),
    "Precision": precision_score(y_test, log_pred, zero_division=0),
    "Recall": recall_score(y_test, log_pred, zero_division=0),
    "F1": f1_score(y_test, log_pred, zero_division=0),
    "PR_AUC": average_precision_score(y_test, log_prob),
    "ROC_AUC": roc_auc_score(y_test, log_prob)
}

display(pd.DataFrame([log_metrics]))
print(classification_report(y_test, log_pred, target_names=["Legitimate", "Fraud"], zero_division=0))


## 5. Logistic Regression confusion matrix


In [ ]:
cm = confusion_matrix(y_test, log_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)
plt.title("Logistic Regression — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 6. Random Forest comparison

Random Forest can capture nonlinear relationships and interactions between transaction variables.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred, zero_division=0),
    "Recall": recall_score(y_test, rf_pred, zero_division=0),
    "F1": f1_score(y_test, rf_pred, zero_division=0),
    "PR_AUC": average_precision_score(y_test, rf_prob),
    "ROC_AUC": roc_auc_score(y_test, rf_prob)
}

results = pd.DataFrame([log_metrics, rf_metrics])
display(results)


In [ ]:
results.set_index("Model")[["Precision", "Recall", "F1", "PR_AUC"]].plot(
    kind="bar", figsize=(11, 6)
)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Fraud Detection Model Comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=.25)
plt.show()


**Why PR-AUC?** When fraud is rare, Precision-Recall curves focus directly on performance for the positive/fraud class. This makes PR-AUC particularly useful alongside recall and precision.


## 7. Precision-Recall curves


In [ ]:
log_precision, log_recall, _ = precision_recall_curve(y_test, log_prob)
rf_precision, rf_recall, _ = precision_recall_curve(y_test, rf_prob)

plt.figure(figsize=(9, 6))
plt.plot(log_recall, log_precision, label=f"Logistic Regression (AP={log_metrics['PR_AUC']:.3f})")
plt.plot(rf_recall, rf_precision, label=f"Random Forest (AP={rf_metrics['PR_AUC']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 8. Random Forest feature importance


In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance.head(15))

plt.figure(figsize=(10, 7))
sns.barplot(data=importance.head(15), x="Importance", y="Feature")
plt.title("Top Random Forest Features")
plt.xlabel("Importance")
plt.ylabel("")
plt.show()


## 9. Fraud-detection threshold discussion

The default probability threshold is 0.50. In real fraud detection, this threshold should be tuned according to business costs:
- Missing a fraud transaction can be very expensive.
- Flagging a legitimate transaction creates customer friction.

A production system can therefore choose a lower threshold to improve recall, accepting that precision may decrease.


In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

threshold_rows = []
for t in thresholds:
    pred_t = (rf_prob >= t).astype(int)
    threshold_rows.append({
        "Threshold": t,
        "Precision": precision_score(y_test, pred_t, zero_division=0),
        "Recall": recall_score(y_test, pred_t, zero_division=0),
        "F1": f1_score(y_test, pred_t, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

threshold_df.set_index("Threshold")[["Precision", "Recall", "F1"]].plot(
    marker="o", figsize=(10, 6)
)
plt.ylim(0, 1)
plt.title("Threshold Trade-off — Random Forest")
plt.ylabel("Score")
plt.grid(alpha=.25)
plt.show()


## Conclusion

This project demonstrates an end-to-end fraud-detection workflow with a highly imbalanced target.

### Key points
- Accuracy alone is not a reliable fraud metric.
- Precision measures how many flagged transactions are actually fraud.
- Recall measures how much of the actual fraud the model catches.
- F1 balances precision and recall.
- PR-AUC is especially useful for rare-event classification.
- Threshold selection should reflect the financial and customer-experience cost of false positives and false negatives.

### Production considerations
A real payment system would also require time-aware validation, monitoring for concept drift, explainability, human review for high-risk cases, and strict security/privacy controls.


In [ ]:
# Save model evaluation outputs
results.to_csv("../output/model_comparison.csv", index=False)
importance.to_csv("../output/feature_importance.csv", index=False)
threshold_df.to_csv("../output/threshold_analysis.csv", index=False)

predictions = pd.DataFrame({
    "Actual": y_test.values,
    "Logistic_Predicted": log_pred,
    "Logistic_Probability": log_prob,
    "RandomForest_Predicted": rf_pred,
    "RandomForest_Probability": rf_prob
})
predictions.to_csv("../output/test_predictions.csv", index=False)

print("Fraud Detection task completed.")
